# Imports/Settings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# 1. Стандартная библиотека
import sys
import warnings
from pathlib import Path

# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error, r2_score
from hydra import initialize, compose

# 3. Локальные модули (Движок)
sys.path.append(str(Path.cwd().parent))
from src.core.data import UniversalDataLoader
from src.core.pipeline import MLPipeline

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
with initialize(version_base=None, config_path="../configs"):
    # Переопределяем параметры, если нужно (например, принудительно ставим CatBoost)
    cfg = compose(config_name="config", overrides=["model=catboost"])

print(f"Проект: {cfg.project_name} | Режим: Baseline Modeling")

try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Baseline Modeling


# Data Loading

In [ ]:
loader = UniversalDataLoader(cfg)
df = loader.load_data()

# Наш метод get_splits сам разобьет данные согласно config.data.test_size
train_df, val_df, test_df = loader.get_splits(df)

target = cfg.data.tabular.target_col

X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_val, y_val = val_df.drop(columns=[target]), val_df[target]
X_test, y_test = test_df.drop(columns=[target]), test_df[target]

print(f"Обучающая выборка: {X_train.shape}")
print(f"Валидационная выборка: {X_val.shape}")
print(f"Тестовая выборка: {X_test.shape}")

# Baselinge

In [ ]:
# Настраиваем подключение к серверу (можно локально: "file:../mlruns")
mlflow.set_tracking_uri(cfg.logging.mlflow.tracking_uri)
mlflow.set_experiment(cfg.logging.mlflow.experiment_name)

run_name = f"baseline_{cfg.model.name}_v{cfg.model.version}"

# Открываем сессию (Паттерн Менеджера)
with mlflow.start_run(run_name=run_name):
    print(f"Запуск MLflow Run: {run_name}")

    # Инициализируем наш оркестратор пайплайна
    pipeline = MLPipeline(cfg)

    # Запускаем обучение.
    # save_artifacts=True заставит пайплайн сохранить модель и препроцессор в папку models/
    model = pipeline.train(X_train, y_train, X_val, y_val, save_artifacts=True)

    print("Обучение завершено! Артефакты сохранены.")

# Base analise

In [ ]:
# 1. Прогоняем тестовые данные через сохраненный препроцессор
# (Чтобы не писать это руками, в будущем добавим метод evaluate в MLPipeline,
# но для анализа в ноутбуке делаем руками)
prep = pipeline._get_preprocessor()
# ВАЖНО: препроцессор уже обучен внутри pipeline.train, поэтому тут только transform!
X_test_clean = prep.transform(X_test)

# 2. Получаем предсказания
preds = model.predict(X_test_clean)

task_type = cfg.data.tabular.task_type
print(f"--- МЕТРИКИ БЕЙЗЛАЙНА ({task_type.upper()}) ---\n")

if task_type in ['binary', 'multiclass']:
    print(classification_report(y_test, preds))

    if hasattr(model, 'predict_proba') and task_type == 'binary':
        probs = model.predict_proba(X_test_clean)[:, 1]
        auc = roc_auc_score(y_test, probs)
        print(f"ROC-AUC Score: {auc:.4f}")

elif task_type == 'regression':
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"RMSE: {np.sqrt(mse):.4f}")
    print(f"R2 Score: {r2:.4f}")

# Feature Importance

In [ ]:
# CatBoost/LightGBM/XGBoost умеют отдавать важность фичей из коробки
if hasattr(model.model, 'feature_importances_'):
    importances = model.model.feature_importances_
    # Имена колонок берем из очищенного датасета (чтобы учесть OneHot/Target кодирование)
    feature_names = prep.get_feature_names_out() if hasattr(prep, 'get_feature_names_out') else X_test.columns

    fi_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    plt.figure()
    sns.barplot(
        data=fi_df.head(15), # Показываем топ-15
        x='Importance',
        y='Feature',
        palette='viridis',
        alpha=PLOT_ALPHA
    )
    plt.title(f"Топ-15 самых важных признаков ({cfg.model.name})")
    plt.tight_layout()
    plt.show()
else:
    print("У выбранной модели нет встроенного атрибута feature_importances_.")